# 🎙️ Student 1 — Audio Analyst: 911 Call Transcription & Analysis

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Role:** Student 1 — Audio Analyst  
**Pipeline:** Transcribe 911 audio → Extract keywords/entities → Sentiment & urgency scoring → Export CSV

### ✅ Required Output Columns
`Call_ID | Transcript | Extracted_Event | Location | Sentiment | Urgency_Score`

---

## How this notebook works
1. **Cells 1–3** — Fork from Kaggle Wav2Vec2 notebook; installs & transcription already done there. This notebook uses **Whisper** as an equivalent local approach with synthetic 911 audio.
2. **Cells 4–6** — Your own analysis cells (keyword extraction, sentiment, urgency).
3. **Cell 7** — Save the final CSV.

> **Kaggle users:** Run the Wav2Vec2 fork first, then paste Cells 4–7 at the bottom of that notebook. Replace `transcripts` with whatever variable the Wav2Vec2 notebook produces.

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
# Run this once. On Kaggle with GPU this takes ~2-3 minutes.

!pip install openai-whisper spacy transformers torch torchaudio gtts pydub -q
!python -m spacy download en_core_web_sm -q

print("✅ All packages installed.")

In [ ]:
# ============================================================
# CELL 2 — Generate synthetic 911 audio clips (gTTS)
# ============================================================
# Simulates real 911 calls for testing purposes.
# On Kaggle, the Wav2Vec2 notebook provides real audio — skip
# this cell and use 'results' from that notebook instead.

import os
from gtts import gTTS

calls = [
    ("C001", "There is a fire, people are trapped on the second floor of the building on downtown avenue. Please hurry!"),
    ("C002", "I just witnessed a car accident on Main Street near the intersection of Oak Road. Two vehicles, someone is injured."),
    ("C003", "There is a robbery happening right now at the convenience store on Elm Street. The suspect has a weapon."),
    ("C004", "Help! There is a man attacking someone outside the park on River Road. Please send police immediately!"),
    ("C005", "Smoke is coming from the warehouse near the docks on Harbor Boulevard. It looks like it might be on fire."),
]

os.makedirs("audio_files", exist_ok=True)
for call_id, text in calls:
    tts = gTTS(text=text, lang='en')
    tts.save(f"audio_files/{call_id}.mp3")
    print(f"✅ Saved {call_id}.mp3")

print(f"\n{len(calls)} audio files ready in ./audio_files/")

In [ ]:
# ============================================================
# CELL 3 — Transcribe audio using OpenAI Whisper
# ============================================================
# Whisper 'base' model (~140MB) runs well on free Kaggle GPU.
# On Kaggle Wav2Vec2 fork: skip this and use their 'results' variable.

import whisper

model = whisper.load_model("base")  # options: tiny, base, small, medium, large
print("🔊 Whisper model loaded. Transcribing...\n")

# 'transcripts' will be a dict: {Call_ID: transcript_text}
transcripts = {}
for call_id, _ in calls:
    result = model.transcribe(f"audio_files/{call_id}.mp3")
    transcripts[call_id] = result["text"].strip()
    print(f"{call_id}: {transcripts[call_id]}")

print(f"\n✅ Transcribed {len(transcripts)} calls.")

# -----------------------------------------------------------
# 👆 IF USING THE KAGGLE WAV2VEC2 NOTEBOOK:
# Replace the dict above with the notebook's output like this:
#
#   transcripts = {f"C{i+1:03d}": text for i, text in enumerate(results)}
#
# where 'results' is whatever variable holds the transcribed strings.
# -----------------------------------------------------------

In [ ]:
# ============================================================
# CELL A — spaCy: keyword & named-entity extraction
# ============================================================
# Extracts locations (GPE / LOC / FAC) and classifies incident
# type using rule-based keyword matching.

import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

KEYWORD_MAP = [
    (["fire", "flame", "smoke", "burning", "blaze"],           "Fire / Building fire"),
    (["accident", "crash", "collision", "vehicle", "car"],     "Road accident"),
    (["robbery", "theft", "stolen", "gun", "weapon", "steal"], "Robbery / Theft"),
    (["fight", "assault", "attack", "violence", "attacking"],  "Assault / Fight"),
    (["trapped", "injured", "hurt", "medical", "unconscious"], "Medical Emergency"),
]

def classify_event(text):
    text_lower = text.lower()
    for keywords, label in KEYWORD_MAP:
        if any(w in text_lower for w in keywords):
            return label
    return "General incident"

def extract_location(text):
    doc = nlp(text)
    locs = [ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC", "FAC")]
    return locs[0] if locs else "Unknown"

# Build the base dataframe
rows = []
for call_id, transcript in transcripts.items():
    rows.append({
        "Call_ID":         call_id,
        "Transcript":      transcript[:300],          # cap at 300 chars for readability
        "Extracted_Event": classify_event(transcript),
        "Location":        extract_location(transcript),
    })

audio_df = pd.DataFrame(rows)
print("📋 Extraction complete:")
print(audio_df[["Call_ID", "Extracted_Event", "Location"]].to_string(index=False))

In [ ]:
# ============================================================
# CELL B — Sentiment & urgency analysis
# ============================================================
# Sentiment: DistilBERT fine-tuned on SST-2 (NEGATIVE = Distressed)
# Urgency:   keyword hit-count + sentiment confidence score

from transformers import pipeline

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

URGENCY_WORDS = [
    "help", "emergency", "hurry", "dying", "trapped", "please",
    "now", "quickly", "fire", "weapon", "injured", "immediately",
    "robbery", "attacking", "shot"
]

sentiments = []
urgencies  = []

for _, row in audio_df.iterrows():
    text = row["Transcript"][:512]          # DistilBERT token limit
    result = sentiment_pipe(text)[0]

    # Sentiment label
    sentiment = "Distressed" if result["label"] == "NEGATIVE" else "Calm"

    # Urgency score: keyword hits (0.1 each) + model confidence (up to 0.4), capped at 1.0
    hit_count = sum(1 for w in URGENCY_WORDS if w in text.lower())
    urgency = round(min(0.2 + (hit_count * 0.1) + result["score"] * 0.3, 1.0), 2)

    sentiments.append(sentiment)
    urgencies.append(urgency)
    print(f"{row['Call_ID']} → Sentiment: {sentiment:10s} | Urgency: {urgency:.2f} | Hits: {hit_count}")

audio_df["Sentiment"]    = sentiments
audio_df["Urgency_Score"] = urgencies

print("\n✅ Sentiment and urgency scoring complete.")

In [ ]:
# ============================================================
# CELL C — Validate & save audio_output.csv
# ============================================================

REQUIRED_COLUMNS = ["Call_ID", "Transcript", "Extracted_Event", "Location", "Sentiment", "Urgency_Score"]

# Ensure correct column order
audio_df = audio_df[REQUIRED_COLUMNS]

# Validate
assert list(audio_df.columns) == REQUIRED_COLUMNS, f"Column mismatch! Got: {list(audio_df.columns)}"
assert len(audio_df) > 0, "DataFrame is empty!"

audio_df.to_csv("audio_output.csv", index=False)

print("✅ Saved audio_output.csv")
print(f"   Rows: {len(audio_df)} | Columns: {list(audio_df.columns)}\n")
print(audio_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL D — Download (Colab only)
# ============================================================
# On Kaggle: use the folder icon on the left sidebar to find
# audio_output.csv and download it manually.
# On Google Colab: uncomment the lines below.

# from google.colab import files
# files.download("audio_output.csv")

print("📥 On Kaggle: click the 📁 folder icon (left sidebar) → find audio_output.csv → download.")
print("📥 On Colab:  uncomment the two lines above.")

---
## 📤 GitHub Upload Instructions

1. **Download `audio_output.csv`** — Kaggle: folder icon → file → Download  
2. **Download this notebook** — Kaggle: `File → Download Notebook`  
3. **Go to your GitHub repo** → open the `/audio` folder  
4. Click **Add file → Upload files**  
5. Drag both `audio_output.csv` and this `.ipynb` file → **Commit changes**

### ✅ Final CSV Column Checklist
| Column | Type | Example |
|---|---|---|
| `Call_ID` | string | `C001` |
| `Transcript` | string (≤300 chars) | `There is a fire...` |
| `Extracted_Event` | string | `Fire / Building fire` |
| `Location` | string | `Downtown Ave` |
| `Sentiment` | `Distressed` / `Calm` | `Distressed` |
| `Urgency_Score` | float 0.0–1.0 | `0.91` |